
# Complete Credit Risk Data Science Project

This notebook covers ALL tasks mentioned in the assignment:
- Conceptual Foundation
- Data Acquisition
- Data Understanding & Cleaning
- Missing Value Handling
- Outlier Handling
- Feature Engineering
- Feature Scaling
- Feature Transformation
- Final Dataset Export
- Final Report

Datasets:
- customer_credit_risk_dataset_12000.csv
- customer_credit_risk_dataset_12000.json
- SQLite DB template included



# Part A: Conceptual Foundation

## What is Data Analysis?
Data Analysis is the process of inspecting, cleaning, transforming, and modeling data to discover useful insights and support decision-making.

## How to Plan a Data Science Project
1. Problem Definition
2. Data Collection
3. Data Cleaning
4. Exploratory Data Analysis
5. Feature Engineering
6. Model Building
7. Evaluation
8. Deployment

## How to Frame a Machine Learning Problem
- Define target variable
- Identify feature variables
- Determine supervised/unsupervised learning
- Choose evaluation metrics
- Split train/test datasets

## Tensors
Tensors are multidimensional arrays used in machine learning.

### NumPy Tensor Examples

In [2]:
import numpy as np

scalar = np.array(10)
vector = np.array([1,2,3])
matrix = np.array([[1,2],[3,4]])
tensor_3d = np.array([[[1],[2]],[[3],[4]]])
print("Scalar:", scalar)
print("Vector:", vector)
print("Matrix:\n", matrix)
print("3D Tensor:\n", tensor_3d)

Scalar: 10
Vector: [1 2 3]
Matrix:
 [[1 2]
 [3 4]]
3D Tensor:
 [[[1]
  [2]]

 [[3]
  [4]]]


In [43]:

import pandas as pd
import numpy as np
import sqlite3
import requests

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    OneHotEncoder,
    Binarizer,
    KBinsDiscretizer,
    StandardScaler,
    MinMaxScaler,
    MaxAbsScaler,
    RobustScaler,
    Normalizer,
    FunctionTransformer,
    PowerTransformer
)

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from scipy import stats

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)


# Part B: Data Acquisition

In [4]:

# Load CSV Dataset

csv_df = pd.read_csv('./Dataset/customer_credit_risk_dataset_12000.csv')

print("CSV Shape:", csv_df.shape)

csv_df.head()


CSV Shape: (12000, 16)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,debt_to_income_ratio,default_flag
0,CUST100000,43.0,Male,South,Secondary,Salaried,237550.99,109491.77,Education,616.0,3,39,53.80,2022-03-04,0.46,0
1,CUST100001,36.0,Female,East,Primary,NaN,783573.67,702947.68,Business,NaN,3,52,30.51,2016-04-01,0.90,0
2,CUST100002,45.0,Male,East,Graduate,Salaried,363902.78,73530.96,Education,756.0,3,56,31.78,2015-04-13,0.20,0
3,CUST100003,54.0,Male,North,Graduate,Unemployed,446099.21,162396.73,Car,665.0,2,41,19.03,2023-04-26,0.36,0
4,CUST100004,35.0,Other,East,Graduate,Salaried,NaN,98621.10,Other,710.0,0,39,38.51,2018-01-31,0.24,0


In [5]:

# Load JSON Dataset

json_df = pd.read_json('./Dataset/customer_credit_risk_dataset_12000.json')

print("JSON Shape:", json_df.shape)

json_df.head()


JSON Shape: (12000, 16)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,debt_to_income_ratio,default_flag
0,CUST100000,43.0,Male,South,Secondary,Salaried,237550.99,109491.77,Education,616.0,3,39,53.80,2022-03-04,0.46,0
1,CUST100001,36.0,Female,East,Primary,None,783573.67,702947.68,Business,NaN,3,52,30.51,2016-04-01,0.90,0
2,CUST100002,45.0,Male,East,Graduate,Salaried,363902.78,73530.96,Education,756.0,3,56,31.78,2015-04-13,0.20,0
3,CUST100003,54.0,Male,North,Graduate,Unemployed,446099.21,162396.73,Car,665.0,2,41,19.03,2023-04-26,0.36,0
4,CUST100004,35.0,Other,East,Graduate,Salaried,NaN,98621.10,Other,710.0,0,39,38.51,2018-01-31,0.24,0


In [6]:
conn = sqlite3.connect("./Dataset/customer_credit_risk_dataset_12000.db")
sql_df = pd.read_sql_query("SELECT * FROM customer_credit_risk LIMIT 5", conn)
print(sql_df)

  customer_id   age  gender region education_level employment_type  \
0  CUST100000  43.0    Male  South       Secondary        Salaried   
1  CUST100001  36.0  Female   East         Primary            None   
2  CUST100002  45.0    Male   East        Graduate        Salaried   
3  CUST100003  54.0    Male  North        Graduate      Unemployed   
4  CUST100004  35.0   Other   East        Graduate        Salaried   

   annual_income  loan_amount loan_purpose  credit_score  repayment_history  \
0      237550.99    109491.77    Education         616.0                  3   
1      783573.67    702947.68     Business           NaN                  3   
2      363902.78     73530.96    Education         756.0                  3   
3      446099.21    162396.73          Car         665.0                  2   
4            NaN     98621.10        Other         710.0                  0   

   transaction_count  spending_ratio   join_date  debt_to_income_ratio  \
0                 39          

In [44]:
url = "https://api.worldbank.org/v2/country/IND/indicator/FP.CPI.TOTL.ZG?format=json"
api_data = requests.get(url).json()
print(api_data[:1])
print("Dummy API example added.")


[{'page': 1, 'pages': 2, 'per_page': 50, 'total': 66, 'sourceid': '2', 'lastupdated': '2026-04-08'}]
Dummy API example added.


# Part C: Data Understanding & Cleaning

In [8]:

df = csv_df.copy()

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           12000 non-null  object 
 1   age                   11400 non-null  float64
 2   gender                11400 non-null  object 
 3   region                12000 non-null  object 
 4   education_level       12000 non-null  object 
 5   employment_type       11400 non-null  object 
 6   annual_income         11400 non-null  float64
 7   loan_amount           12000 non-null  float64
 8   loan_purpose          12000 non-null  object 
 9   credit_score          11400 non-null  float64
 10  repayment_history     12000 non-null  int64  
 11  transaction_count     12000 non-null  int64  
 12  spending_ratio        12000 non-null  float64
 13  join_date             12000 non-null  object 
 14  debt_to_income_ratio  12000 non-null  float64
 15  default_flag       

In [9]:

df.describe(include='all')


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,debt_to_income_ratio,default_flag
count,12000,11400.000000,11400,12000,12000,11400,1.140000e+04,1.200000e+04,12000,11400.000000,12000.000000,12000.000000,12000.000000,12000,12000.000000,12000.000000
unique,12000,NaN,3,4,4,3,NaN,NaN,5,NaN,NaN,NaN,NaN,3508,NaN,NaN
top,CUST100000,NaN,Male,South,Secondary,Salaried,NaN,NaN,Home,NaN,NaN,NaN,NaN,2016-01-05,NaN,NaN
freq,1,NaN,5809,3086,4236,7403,NaN,NaN,4170,NaN,NaN,NaN,NaN,11,NaN,NaN
mean,NaN,37.616579,NaN,NaN,NaN,NaN,5.501417e+05,2.867276e+05,NaN,678.499561,2.009083,45.072500,28.459958,NaN,0.523919,0.085917
std,NaN,10.613661,NaN,NaN,NaN,NaN,4.665562e+05,2.793401e+05,NaN,73.561916,1.423377,6.727866,15.952589,NaN,0.215031,0.280253
min,NaN,18.000000,NaN,NaN,NaN,NaN,5.679322e+04,1.271941e+04,NaN,250.000000,0.000000,23.000000,0.100000,NaN,0.150000,0.000000
25%,NaN,30.000000,NaN,NaN,NaN,NaN,2.950994e+05,1.259940e+05,NaN,633.000000,1.000000,40.000000,15.940000,NaN,0.340000,0.000000
50%,NaN,37.000000,NaN,NaN,NaN,NaN,4.450095e+05,2.140348e+05,NaN,679.000000,2.000000,45.000000,26.200000,NaN,0.520000,0.000000
75%,NaN,45.000000,NaN,NaN,NaN,NaN,6.703504e+05,3.620697e+05,NaN,726.000000,3.000000,50.000000,39.080000,NaN,0.710000,0.000000


In [45]:

# Pandas Profiling

from ydata_profiling import ProfileReport
profile = ProfileReport(df, explorative=True)
profile.to_file("data_quality_report.html")

print("Pandas Profiling section added.")


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 16.02it/s]

Pandas Profiling section added.


In [11]:

# Missing Values

missing_values = df.isnull().sum()

missing_values[missing_values > 0]


age                600
gender             600
employment_type    600
annual_income      600
credit_score       600
dtype: int64

In [12]:

# Separate Columns

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(num_cols)
print(cat_cols)


['age', 'annual_income', 'loan_amount', 'credit_score', 'repayment_history', 'transaction_count', 'spending_ratio', 'debt_to_income_ratio', 'default_flag']
['customer_id', 'gender', 'region', 'education_level', 'employment_type', 'loan_purpose', 'join_date']


## Missing Value Handling Methods

In [13]:

# Simple Imputer - Numerical

median_imputer = SimpleImputer(strategy='median')

df[num_cols] = median_imputer.fit_transform(df[num_cols])

print("Median imputation completed.")


Median imputation completed.


In [14]:

# Simple Imputer - Categorical

mode_imputer = SimpleImputer(strategy='most_frequent')

df[cat_cols] = mode_imputer.fit_transform(df[cat_cols])

print("Categorical mode imputation completed.")


Categorical mode imputation completed.


In [15]:

# Most Frequent Category Imputation

for col in cat_cols:
    most_frequent = df[col].mode()[0]
    df[col].fillna(most_frequent, inplace=True)

print("Most frequent category imputation completed.")


Most frequent category imputation completed.


In [16]:

# Missing Indicator + Random Sample Imputation

random_sample_df = csv_df.copy()

for col in num_cols:
    
    if random_sample_df[col].isnull().sum() > 0:
        
        random_sample_df[col + '_missing'] = np.where(
            random_sample_df[col].isnull(), 1, 0
        )
        
        random_sample = random_sample_df[col].dropna().sample(
            random_sample_df[col].isnull().sum(),
            random_state=42
        )
        
        random_sample.index = random_sample_df[
            random_sample_df[col].isnull()
        ].index
        
        random_sample_df.loc[
            random_sample_df[col].isnull(), col
        ] = random_sample

print("Random sample imputation completed.")


Random sample imputation completed.


In [17]:

# KNN Imputer

knn_df = csv_df.select_dtypes(include=np.number)

knn_imputer = KNNImputer(n_neighbors=5)

knn_imputed = pd.DataFrame(
    knn_imputer.fit_transform(knn_df),
    columns=knn_df.columns
)

knn_imputed.head()


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,debt_to_income_ratio,default_flag
0,43.0,237550.990,109491.77,616.0,3.0,39.0,53.80,0.46,0.0
1,36.0,783573.670,702947.68,660.8,3.0,52.0,30.51,0.90,0.0
2,45.0,363902.780,73530.96,756.0,3.0,56.0,31.78,0.20,0.0
3,54.0,446099.210,162396.73,665.0,2.0,41.0,19.03,0.36,0.0
4,35.0,382170.464,98621.10,710.0,0.0,39.0,38.51,0.24,0.0


In [18]:

# MICE Algorithm

mice_df = csv_df.select_dtypes(include=np.number)

mice_imputer = IterativeImputer(random_state=42)

mice_imputed = pd.DataFrame(
    mice_imputer.fit_transform(mice_df),
    columns=mice_df.columns
)

mice_imputed.head()


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,debt_to_income_ratio,default_flag
0,43.0,237550.990000,109491.77,616.000000,3.0,39.0,53.80,0.46,0.0
1,36.0,783573.670000,702947.68,696.167047,3.0,52.0,30.51,0.90,0.0
2,45.0,363902.780000,73530.96,756.000000,3.0,56.0,31.78,0.20,0.0
3,54.0,446099.210000,162396.73,665.000000,2.0,41.0,19.03,0.36,0.0
4,35.0,290150.396454,98621.10,710.000000,0.0,39.0,38.51,0.24,0.0


In [19]:

# Complete Case Analysis

complete_case_df = csv_df.dropna()

print("Original Shape:", csv_df.shape)
print("After Complete Case Analysis:", complete_case_df.shape)


Original Shape: (12000, 16)
After Complete Case Analysis: (9270, 16)


# Part D: Outlier Handling

In [20]:

# Z-Score Method

z_scores = np.abs(stats.zscore(df[num_cols]))

z_outliers = (z_scores > 3).sum()

print(z_outliers)


1554


In [21]:

# IQR Method

Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)

IQR = Q3 - Q1

iqr_outliers = (
    ((df[num_cols] < (Q1 - 1.5 * IQR)) |
    (df[num_cols] > (Q3 + 1.5 * IQR)))
).sum()

print(iqr_outliers)


age                      111
annual_income            666
loan_amount              612
credit_score             109
repayment_history         64
transaction_count         29
spending_ratio            58
debt_to_income_ratio       0
default_flag            1031
dtype: int64


In [22]:

# Percentile Method

lower = df[num_cols].quantile(0.01)
upper = df[num_cols].quantile(0.99)

percentile_outliers = (
    ((df[num_cols] < lower) |
    (df[num_cols] > upper))
).sum()

print(percentile_outliers)


age                     111
annual_income           240
loan_amount             240
credit_score            237
repayment_history        64
transaction_count       217
spending_ratio          240
debt_to_income_ratio    145
default_flag              0
dtype: int64


In [23]:

# Winsorization

for col in num_cols:
    lower_limit = df[col].quantile(0.01)
    upper_limit = df[col].quantile(0.99)
    
    df[col] = np.clip(df[col], lower_limit, upper_limit)

print("Winsorization completed.")


Winsorization completed.


# Part E: Feature Engineering

In [24]:

# Date Features

df['join_date'] = pd.to_datetime(df['join_date'])

df['year'] = df['join_date'].dt.year
df['month'] = df['join_date'].dt.month
df['day'] = df['join_date'].dt.day
df['weekday'] = df['join_date'].dt.weekday

df[['year', 'month', 'day', 'weekday']].head()


,year,month,day,weekday
0,2022,3,4,4
1,2016,4,1,4
2,2015,4,13,0
3,2023,4,26,2
4,2018,1,31,2


In [25]:

# Ordinal Encoding

ordinal_encoder = OrdinalEncoder(categories=[
    ['Primary', 'Secondary', 'Graduate', 'Post-Graduate']
])

df['education_encoded'] = ordinal_encoder.fit_transform(
    df[['education_level']]
)

df[['education_level', 'education_encoded']].head()


,education_level,education_encoded
0,Secondary,1.0
1,Primary,0.0
2,Graduate,2.0
3,Graduate,2.0
4,Graduate,2.0


In [26]:

# Label Encoding

label_encoder = LabelEncoder()

df['gender_encoded'] = label_encoder.fit_transform(df['gender'])

df[['gender', 'gender_encoded']].head()


,gender,gender_encoded
0,Male,1
1,Female,0
2,Male,1
3,Male,1
4,Other,2


In [27]:

# One-Hot Encoding

df_encoded = pd.get_dummies(
    df,
    columns=['region', 'loan_purpose'],
    drop_first=True
)

df_encoded.head()


,customer_id,age,gender,education_level,employment_type,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,join_date,debt_to_income_ratio,default_flag,year,month,day,weekday,education_encoded,gender_encoded,region_North,region_South,region_West,loan_purpose_Car,loan_purpose_Education,loan_purpose_Home,loan_purpose_Other
0,CUST100000,43.0,Male,Secondary,Salaried,237550.990,109491.77,616.0,3.0,39.0,53.80,2022-03-04,0.46,0.0,2022,3,4,4,1.0,1,False,True,False,False,True,False,False
1,CUST100001,36.0,Female,Primary,Salaried,783573.670,702947.68,679.0,3.0,52.0,30.51,2016-04-01,0.89,0.0,2016,4,1,4,0.0,0,False,False,False,False,False,False,False
2,CUST100002,45.0,Male,Graduate,Salaried,363902.780,73530.96,756.0,3.0,56.0,31.78,2015-04-13,0.20,0.0,2015,4,13,0,2.0,1,False,False,False,False,True,False,False
3,CUST100003,54.0,Male,Graduate,Unemployed,446099.210,162396.73,665.0,2.0,41.0,19.03,2023-04-26,0.36,0.0,2023,4,26,2,2.0,1,True,False,False,True,False,False,False
4,CUST100004,35.0,Other,Graduate,Salaried,445009.505,98621.10,710.0,0.0,39.0,38.51,2018-01-31,0.24,0.0,2018,1,31,2,2.0,2,False,False,False,False,False,False,True


In [28]:

# Binning

df['income_bin'] = pd.cut(
    df['annual_income'],
    bins=5,
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

df[['annual_income', 'income_bin']].head()


,annual_income,income_bin
0,237550.990,Very Low
1,783573.670,Low
2,363902.780,Very Low
3,446099.210,Very Low
4,445009.505,Very Low


In [29]:

# Binarization

binarizer = Binarizer(threshold=500000)

df['income_binary'] = binarizer.fit_transform(
    df[['annual_income']]
)

df[['annual_income', 'income_binary']].head()


,annual_income,income_binary
0,237550.990,0.0
1,783573.670,1.0
2,363902.780,0.0
3,446099.210,0.0
4,445009.505,0.0


In [30]:

# Quantile Binning

df['income_quantile'] = pd.qcut(
    df['annual_income'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

df[['annual_income', 'income_quantile']].head()


,annual_income,income_quantile
0,237550.990,Q1
1,783573.670,Q4
2,363902.780,Q2
3,446099.210,Q3
4,445009.505,Q2


In [31]:

# KMeans Binning

kmeans = KMeans(n_clusters=4, random_state=42)

df['kmeans_income_bin'] = kmeans.fit_predict(
    df[['annual_income']]
)

df[['annual_income', 'kmeans_income_bin']].head()


,annual_income,kmeans_income_bin
0,237550.990,3
1,783573.670,0
2,363902.780,3
3,446099.210,1
4,445009.505,1


# Part F: Feature Scaling

In [32]:

scale_cols = [
    'annual_income',
    'loan_amount',
    'credit_score',
    'transaction_count'
]

# Standardization
standard_scaler = StandardScaler()
standard_scaled = standard_scaler.fit_transform(df[scale_cols])

# Normalization
normalizer = Normalizer()
normalized = normalizer.fit_transform(df[scale_cols])

# MinMax Scaling
minmax_scaler = MinMaxScaler()
minmax_scaled = minmax_scaler.fit_transform(df[scale_cols])

# MaxAbs Scaling
maxabs_scaler = MaxAbsScaler()
maxabs_scaled = maxabs_scaler.fit_transform(df[scale_cols])

# Robust Scaling
robust_scaler = RobustScaler()
robust_scaled = robust_scaler.fit_transform(df[scale_cols])

print("All scaling methods completed.")


All scaling methods completed.


# Part G: Feature Construction & Transformation

In [33]:

# FunctionTransformer

log_transformer = FunctionTransformer(np.log1p)

df['log_income'] = log_transformer.transform(
    df[['annual_income']]
)

df[['annual_income', 'log_income']].head()


,annual_income,log_income
0,237550.990,12.378142
1,783573.670,13.571622
2,363902.780,12.804645
3,446099.210,13.008299
4,445009.505,13.005853


In [34]:

# Reciprocal Transformation

df['reciprocal_income'] = 1 / (df['annual_income'] + 1)

df[['annual_income', 'reciprocal_income']].head()


,annual_income,reciprocal_income
0,237550.990,0.000004
1,783573.670,0.000001
2,363902.780,0.000003
3,446099.210,0.000002
4,445009.505,0.000002


In [35]:

# Square Root Transformation

df['sqrt_income'] = np.sqrt(df['annual_income'])

df[['annual_income', 'sqrt_income']].head()


,annual_income,sqrt_income
0,237550.990,487.392029
1,783573.670,885.196967
2,363902.780,603.243549
3,446099.210,667.906588
4,445009.505,667.090327


In [36]:

# PowerTransformer - Yeo Johnson

yeo_transformer = PowerTransformer(method='yeo-johnson')

df['yeo_income'] = yeo_transformer.fit_transform(
    df[['annual_income']]
)

df[['annual_income', 'yeo_income']].head()


,annual_income,yeo_income
0,237550.990,-1.064737
1,783573.670,0.955776
2,363902.780,-0.335172
3,446099.210,0.010231
4,445009.505,0.006094


In [37]:

# PowerTransformer - Box Cox

positive_income = df[['annual_income']].copy()

positive_income = positive_income + 1

boxcox_transformer = PowerTransformer(method='box-cox')

boxcox_data = boxcox_transformer.fit_transform(positive_income)

print(boxcox_data[:5])


[[-1.06473691]
 [ 0.95577576]
 [-0.33517242]
 [ 0.0102306 ]
 [ 0.00609388]]


In [38]:

# ColumnTransformer Example

numeric_features = [
    'annual_income',
    'loan_amount'
]

categorical_features = [
    'gender',
    'employment_type'
]

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

processed_data = preprocessor.fit_transform(df)

print("ColumnTransformer applied successfully.")


ColumnTransformer applied successfully.


In [39]:

# Feature Construction

df['debt_to_income_feature'] = (
    df['loan_amount'] / (df['annual_income'] + 1)
)

df['average_monthly_transactions'] = (
    df['transaction_count'] / 12
)

df['spending_to_income_ratio'] = (
    df['spending_ratio'] / (df['annual_income'] + 1)
)

df[[
    'debt_to_income_feature',
    'average_monthly_transactions',
    'spending_to_income_ratio'
]].head()


,debt_to_income_feature,average_monthly_transactions,spending_to_income_ratio
0,0.460917,3.250000,0.000226
1,0.897104,4.333333,0.000039
2,0.202062,4.666667,0.000087
3,0.364036,3.416667,0.000043
4,0.221615,3.250000,0.000087


# Part H: Final Deliverable

In [40]:

print("Final Dataset Shape:", df.shape)

df.head()


Final Dataset Shape: (12000, 33)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,debt_to_income_ratio,default_flag,year,month,day,weekday,education_encoded,gender_encoded,income_bin,income_binary,income_quantile,kmeans_income_bin,log_income,reciprocal_income,sqrt_income,yeo_income,debt_to_income_feature,average_monthly_transactions,spending_to_income_ratio
0,CUST100000,43.0,Male,South,Secondary,Salaried,237550.990,109491.77,Education,616.0,3.0,39.0,53.80,2022-03-04,0.46,0.0,2022,3,4,4,1.0,1,Very Low,0.0,Q1,3,12.378142,0.000004,487.392029,-1.064737,0.460917,3.250000,0.000226
1,CUST100001,36.0,Female,East,Primary,Salaried,783573.670,702947.68,Business,679.0,3.0,52.0,30.51,2016-04-01,0.89,0.0,2016,4,1,4,0.0,0,Low,1.0,Q4,0,13.571622,0.000001,885.196967,0.955776,0.897104,4.333333,0.000039
2,CUST100002,45.0,Male,East,Graduate,Salaried,363902.780,73530.96,Education,756.0,3.0,56.0,31.78,2015-04-13,0.20,0.0,2015,4,13,0,2.0,1,Very Low,0.0,Q2,3,12.804645,0.000003,603.243549,-0.335172,0.202062,4.666667,0.000087
3,CUST100003,54.0,Male,North,Graduate,Unemployed,446099.210,162396.73,Car,665.0,2.0,41.0,19.03,2023-04-26,0.36,0.0,2023,4,26,2,2.0,1,Very Low,0.0,Q3,1,13.008299,0.000002,667.906588,0.010231,0.364036,3.416667,0.000043
4,CUST100004,35.0,Other,East,Graduate,Salaried,445009.505,98621.10,Other,710.0,0.0,39.0,38.51,2018-01-31,0.24,0.0,2018,1,31,2,2.0,2,Very Low,0.0,Q2,1,13.005853,0.000002,667.090327,0.006094,0.221615,3.250000,0.000087


In [41]:

# Export Final Dataset

df.to_csv('final_cleaned_credit_risk_dataset.csv', index=False)

print("Final cleaned dataset exported successfully.")


Final cleaned dataset exported successfully.



# Final Report

## Missing Value Handling
- Median Imputation
- Most Frequent Imputation
- Random Sample Imputation
- KNN Imputation
- MICE Algorithm
- Complete Case Analysis

## Outlier Handling
- Z-Score Method
- IQR Method
- Percentile Method
- Winsorization

## Encoding Techniques
- Ordinal Encoding
- Label Encoding
- One-Hot Encoding
- Binarization
- Quantile Binning
- KMeans Binning

## Scaling Methods
- StandardScaler
- Normalizer
- MinMaxScaler
- MaxAbsScaler
- RobustScaler

## Transformations
- Log Transformation
- Reciprocal Transformation
- Square Root Transformation
- Yeo-Johnson
- Box-Cox

## New Features Created
- Debt-to-Income Feature
- Average Monthly Transactions
- Spending-to-Income Ratio

Dataset is fully cleaned and ready for Machine Learning.
